# 06. 구독 안내 시점 A/B 테스트 설계 및 실행

## 1. 기본 실험 설계

**가설:** 첫 완료 후 표시하던 구독 안내를 두 번째 레슨 완료 이후로 미루면, 7일 내 두 번째 레슨 시작률이 높아질 것이다.

| 항목 | 설계 초안 |
| --- | --- |
| 대상 | 첫 레슨을 시작 후 24시간 이내 완료한 신규 영어 학습자 |
| 배정 시점 | 첫 완료 직후, 구독 안내 표시 전 |
| 배정 방식 | 사용자 단위로 50:50 무작위 배정하고 배정 유지 |
| A군: 대조군 | 기존처럼 첫 레슨 완료 후 구독 안내 |
| B군: 실험군 | 두 번째 레슨 완료 후 같은 구독 안내 |
| 유지할 조건 | 구독 가격·문구·혜택, 에너지 보상, 문제 구성 |
| 주요 지표 | 배정 후 7일 내 두 번째 레슨 시작률 |
| 보조 학습 지표 | 배정 후 7일 내 두 번째 레슨 완료율 |
| 사업 확인 지표 | 배정 후 14일 내 유료구독 전환율 |


각 군에 배정된 전체 사용자로 설정       
B군에서 두 번째 레슨을 완료하지 못해 안내를 보지 않은 사용자도 포함한다.

## 2. 라이브러리 및 데이터 호출

04번에서 저장한 `funnel_users.csv`를 재사용. 기존 프로젝트의 `numpy`, `pandas`, `scipy`로 계산한다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm


In [2]:
ROOT = Path(r"C:\Users\joon5\Documents\github\duolingo-funnel-analysis")
DATA_PROCESSED_DIR = ROOT / "data" / "processed"
FUNNEL_USERS_PATH = DATA_PROCESSED_DIR / "funnel_users.csv"

funnel_users = pd.read_csv(FUNNEL_USERS_PATH)
completed_users = funnel_users[funnel_users["step_4"]].copy()

# 기존 완료자의 후속 시작률을 표본 계획용 기준률로 사용
baseline_rate = completed_users["step_5"].mean()

print("전체 사용자:", len(funnel_users), "명")
print("첫 레슨 완료자:", len(completed_users), "명")
print("두 번째 시작자:", int(completed_users["step_5"].sum()), "명")
print("계획용 기준 시작률:", round(baseline_rate * 100, 2), "%")


전체 사용자: 10000 명
첫 레슨 완료자: 6485 명
두 번째 시작자: 2815 명
계획용 기준 시작률: 43.41 %


## 3. 표본 수 계산 기준

| 항목 | 이번 계획의 설정 |
| --- | --- |
| 기준 시작률 | 기존 전체 완료자의 약 43.41% |
| 최소 탐지 차이(MDE) | +5%p를 기본안으로 사용하고 +1·+3%p도 비교 |
| 유의수준 | 5%, 양측 검정 |
| 검정력 | 80% |
| 배정 비율 | A:B = 1:1 |

- **MDE:** 실험이 포착할 수 있도록 계획하는 차이의 크기. 예상 성과를 뜻하지 않음.
- **검정력:** 가정한 차이가 실제로 있을 때 이를 통계적으로 확인할 확률을 계획하는 기준.
- 43.41%에서 +5%p는 48.41%가 되는 차이. 사업상 최소 이득이나 자동 적용 기준과는 별개.
- 기존 구독 노출·미노출 집단의 차이는 이 실험의 예상 효과로 사용하지 않음.


In [ ]:
alpha = 0.05
power = 0.80
target_mde_pp = 5

# 구독 지표는 기준 전환율과 사업상 허용 감소폭을 확인한 뒤 입력
paid_baseline_rate = None
paid_allowed_drop_pp = None

### 3.1 확인하려는 차이별 필요 인원

두 독립 비율의 정규근사 공식을 사용. 기준률과 변경 후 비율의 분산, 유의수준, 검정력을 반영하여 군별 인원을 계산한다.
이 값은 주지표 기준의 근사치이며, 유료구독 지표에 필요한 표본은 아직 포함하지 않는다.


In [4]:
z_alpha = norm.ppf(1 - alpha / 2)
z_power = norm.ppf(power)

sample_rows = []

for mde_pp in [1, 3, 5]:
    difference = mde_pp / 100
    rate_a = baseline_rate
    rate_b = baseline_rate + difference
    average_rate = (rate_a + rate_b) / 2

    # 귀무가설·대립가설 아래의 표준편차를 반영한 동수 배정 근사식
    required_per_group = (
        z_alpha * np.sqrt(2 * average_rate * (1 - average_rate))
        + z_power * np.sqrt(rate_a * (1 - rate_a) + rate_b * (1 - rate_b))
    ) ** 2 / difference ** 2

    required_per_group = int(np.ceil(required_per_group))
    sample_rows.append([mde_pp, required_per_group, required_per_group * 2])

sample_plan = pd.DataFrame(sample_rows, columns=[
    "가정한 차이(%p)", "군별 필요 인원(명)", "전체 필요 인원(명)"
])
display(sample_plan)


,가정한 차이(%p),군별 필요 인원(명),전체 필요 인원(명)
0,1,38661,77322
1,3,4315,8630
2,5,1559,3118


### 3.2 이번 초안의 모집 인원

+5%p 조건의 군별 필요 인원을 100명 단위로 올림하여 계획한다.
최종 규모는 주지표와 구독 지표에 필요한 표본 중 더 큰 값을 반영하여 확정한다.


In [5]:
required_per_group = int(sample_plan.loc[
    sample_plan["가정한 차이(%p)"] == target_mde_pp, "군별 필요 인원(명)"
].iloc[0])

planned_per_group = int(np.ceil(required_per_group / 100) * 100)
planned_total = planned_per_group * 2

group_plan = pd.DataFrame({
    "집단": ["A군: 대조군", "B군: 실험군"],
    "계획 인원(명)": [planned_per_group, planned_per_group],
})
display(group_plan)
print("전체 계획 인원:", planned_total, "명")
print("주지표만을 기준으로 한 표본 계획입니다.")


전체 계획 인원: 3200 명
주지표만을 기준으로 한 표본 계획입니다.


,집단,계획 인원(명)
0,A군: 대조군,1600
1,B군: 실험군,1600


## 4. 모집 기간과 관찰 기간

일일 배정 가능 인원은 실제 서비스 유입량을 확인해야 한다. 아래 50·100·200명은 계산 방법을 보여주는 가정이다.

`모집 기간 = 전체 계획 인원 ÷ 하루 배정 가능 인원`을 올림하되, 이번 초안에서는 최소 14일로 계획한다.
마지막 배정자의 학습 지표는 7일, 구독 지표는 14일을 관찰하므로 전체 기간에는 모집 종료 후 14일을 더한다.


In [6]:
min_enrollment_days = 14
learning_observation_days = 7
paid_observation_days = 14

duration_plan = pd.DataFrame({"하루 배정 가능 인원(가정)": [50, 100, 200]})
enrollment_days = np.ceil(
    planned_total / duration_plan["하루 배정 가능 인원(가정)"]
).astype(int)

duration_plan["모집 기간(일)"] = np.maximum(enrollment_days, min_enrollment_days)
duration_plan["마지막 사용자 관찰(일)"] = paid_observation_days
duration_plan["전체 기간 예시(일)"] = (
    duration_plan["모집 기간(일)"] + duration_plan["마지막 사용자 관찰(일)"]
)
display(duration_plan)


,하루 배정 가능 인원(가정),모집 기간(일),마지막 사용자 관찰(일),전체 기간 예시(일)
0,50,64,14,78
1,100,32,14,46
2,200,16,14,30


## 5. 성공·중단 기준 초안

| 항목 | 계획 |
| --- | --- |
| 주검정 | 두 비율 차이의 양측 z 검정, 유의수준 0.05 |
| 효과 제시 | B−A 시작률 차이(%p)와 95% 신뢰구간 |
| 학습 개선 근거 | B군 시작률이 더 높고 주검정 p < 0.05. 차이 크기와 신뢰구간도 함께 해석 |
| 불확실한 결과 | 차이를 확인하지 못하면 효과가 정확히 0이라고 단정하지 않음 |
| 사업 영향 | 14일 유료구독 전환율의 허용 감소폭을 사전 설정하고 그 범위 내인지 확인 |
| 종료 원칙 | 정해진 모집·관찰 기간 종료 후 판단. 중간 p값을 이유로 종료 시점을 바꾸지 않음 |
| 운영 중 확인 | 배정 중복·배정 비율·로그 장애·심각한 지표 악화를 확인하고 필요 시 중단·원인 조사 |

사업 지표의 기준률·허용 감소폭과 그 표본 계획은 미정이다. 이를 정하기 전에는 전면 적용 여부를 확정하지 않는다.
